#### 01. Write a query that returns total revenue (Quantity × Price) and total quantity sold, grouped by Country, sorted highest revenue first — excluding cancelled orders.

In [0]:
from pyspark.sql import functions as f
# from pyspark.sql import desc

df = spark.table("retail_sales_dev.silver.sales_cleaned").select("Country","Quantity","Price","Is_Cancelled").filter(f.col("Is_Cancelled") == False)

df_grouped = df.groupBy("Country").agg(
    f.sum(f.col("Quantity") * f.col("Price")).alias("TotalRevenue"),
    f.sum(f.col("Quantity")).alias("TotalQuantity")
).orderBy(f.desc("TotalRevenue"))

display(df_grouped) 
# display(df)

In [0]:
%sql

SELECT COUNTRY,SUM(QUANTITY*PRICE) AS TotalRevenue,
SUM(QUANTITY) AS TotalQuantity
FROM retail_sales_dev.silver.sales_cleaned
WHERE is_cancelled == FALSE -- OR NOT is_cancelled
group by COUNTRY
order by TotalRevenue DESC;


#### 02. find the customer with the 3rd highest total spend overall — not per country, just globally. Write it in SQL, then PySpark.

In [0]:
%sql
SELECT CustomerID FROM(
SELECT customerid,SUM(Quantity * PRICE),dense_rank() over(ORDER BY SUM(Quantity * PRICE) DESC) AS RN FROM retail_sales_dev.silver.sales_cleaned
WHERE NOT is_cancelled AND CustomerID <> 'UNKNOWN'
group by CustomerID)
WHERE RN = 3


In [0]:
from pyspark.sql.window import Window

df = spark.table("retail_sales_dev.silver.sales_cleaned").filter((f.col("Is_Cancelled") == False) & (f.col("CustomerID")!='UNKNOWN')).select("Country","CustomerID","Quantity","Price")

df_grouped = df.groupBy("CustomerID").agg(
    f.sum(f.col("Quantity") * f.col("Price")).alias("TotalSpend"))
   
# w = Window.orderBy(f.col("TotalSpend").desc())
w = Window.partitionBy(f.col("CustomerID")).orderBy(f.col("TotalSpend").desc())
df_rank = df_grouped.withColumn("rank", f.dense_rank().over(w))

df_filtered = df_rank.select(f.col("CustomerID")).filter(f.col("rank")==3)

display(df_filtered)

#### 03 Joins

In [0]:
from pyspark.sql import functions as f

df_sales = spark.read.table("retail_sales_dev.gold.fact_sales")
# .filter(f.col("CustomerSK").isNull()) ##76852
df_customer = spark.read.table("retail_sales_dev.gold.dim_customer")

df = df_sales.join(df_customer, on="CustomerSK", how="inner")

display(df_sales.count())


#### 05 For each customer, find their single highest-value order (by Invoice, summing Quantity × Price across the line items of that invoice) — i.e., rank each customer's own invoices by value and keep only their no.1. Exclude cancelled orders and UNKNOWN customers.

In [0]:
# # from pyspark.sql import functions as f
from pyspark.sql.window import Window

df = spark.read.table("retail_sales_dev.silver.sales_cleaned").select(
    "CustomerID",
    "Invoice", (f.col("Quantity") * f.col("Price")).alias("TotalAmount")
).filter((f.col("CustomerID")!='UNKNOWN')& (f.col("is_cancelled") == False))

df_grouped = df.groupBy("CustomerID","Invoice").agg(
    f.sum(f.col("TotalAmount")).alias("TotalSpend")
)

w = Window.partitionBy(f.col("CustomerID")).orderBy(f.col("TotalSpend").desc())

df_ranked = df_grouped.withColumn("rank", f.row_number().over(w))

df_filtered = df_ranked.select("CustomerID","Invoice").filter(f.col("rank")==1)

display(df_filtered)

# df = spark.read.table("retail_sales_dev.silver.sales_cleaned").filter((f.col("CustomerID")!='UNKNOWN') & (f.col("CustomerID") == '14309'))

# display(df)

In [0]:
%sql

-- with CTE AS (Select CustomerID,Invoice,Price*Quantity as TotalAmount from retail_sales_dev.silver.sales_cleaned
-- where CustomerID <>'UNKNOWN' and CustomerID = '14309'),
with CTE AS (
Select CustomerID,Invoice,sum(Price * Quantity) TotalAmount from retail_sales_dev.silver.sales_cleaned
where CustomerID <> 'UNKNOWN' and not is_cancelled
group by CustomerID,Invoice
),
CTE2 AS
(select *, dense_rank() over(partition by customerid order by TotalAmount desc) rn from CTE)
Select CustomerID, Invoice from CTE2 where rn = 1
